# Day 5: Baseline Regression Models

Trains a plain Linear Regression baseline on both datasets — no tuning, no feature engineering. The point of a baseline isn't to be good, it's to establish a reference number that any future, more advanced model has to beat.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def train_baseline(df, target, feature_cols, categorical_cols, label):
    print(f"\n{'=' * 50}")
    print(f"Baseline model: {label}")
    print("=" * 50)

    X = df[feature_cols]
    y = df[target]

    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, predictions)

    print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
    print(f"MAE:  {mae:.2f}")
    print(f"MSE:  {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2:   {r2:.4f}")

    coefficients = pd.Series(model.coef_, index=X.columns)
    coefficients = coefficients.reindex(coefficients.abs().sort_values(ascending=False).index)
    print("\nTop 5 most influential features:")
    print(coefficients.head(5))

    return model

## Auto MPG

Predicting `mpg` from a car's specs (cylinders, displacement, horsepower, weight, acceleration, model year, origin).

In [2]:
mpg_df = pd.read_csv("../Datasets/auto_mpg_cleaned.csv")

mpg_model = train_baseline(
    mpg_df,
    target="mpg",
    feature_cols=["cylinders", "displacement", "horsepower", "weight",
                  "acceleration", "model_year", "origin"],
    categorical_cols=["origin"],
    label="Auto MPG (predicting mpg)",
)


Baseline model: Auto MPG (predicting mpg)
Train rows: 313, Test rows: 79
MAE:  2.46
MSE:  10.60
RMSE: 3.26
R2:   0.7923

Top 5 most influential features:
origin_usa     -2.875499
model_year      0.797161
cylinders      -0.342101
origin_japan    0.330470
acceleration    0.042198
dtype: float64


## Houston Housing

Predicting `price` from beds, baths, area, location, tax value, lot size, and home type.

Note: `zestimate` (Zillow's own price estimate) is deliberately excluded from the features. Including it would be a data leakage bug — it's essentially another estimate of the exact value we're trying to predict.

In [3]:
housing_df = pd.read_csv("../Datasets/houston_housing_cleaned.csv")

housing_model = train_baseline(
    housing_df,
    target="price",
    feature_cols=["beds", "baths", "area", "latitude", "longitude",
                  "tax_assessed_value", "lot_area_value", "days_on_zillow",
                  "home_type"],
    categorical_cols=["home_type"],
    label="Houston Housing (predicting price)",
)


Baseline model: Houston Housing (predicting price)
Train rows: 320, Test rows: 80
MAE:  102454.25
MSE:  25767097423.40
RMSE: 160521.33
R2:   0.7874

Top 5 most influential features:
baths                      36496.251810
home_type_SINGLE_FAMILY    23758.449273
beds                      -18318.136838
home_type_TOWNHOUSE       -12112.875388
home_type_MULTI_FAMILY     -8550.503298
dtype: float64


## Summary

| Dataset | MAE | RMSE | R² |
| --- | --- | --- | --- |
| Auto MPG (mpg) | 2.46 | 3.26 | 0.79 |
| Houston Housing (price) | $102,454 | $160,521 | 0.79 |

Both baselines explain about 79% of the variance in their target. Neither is tuned or optimized — that's intentional. These numbers are the floor that Week 2's more advanced models (Ridge, Lasso, Polynomial, and beyond) need to beat to justify their added complexity.